In [ ]:
import numpy as np
from helpers import *

DATA_PATH = "./dataset/"

In [ ]:
x_train, x_test, y_train, train_ids, test_ids = load_csv_data("./dataset")


In [ ]:
data = np.genfromtxt(
    "./dataset/x_train.csv",
    delimiter=",",
    names=True,
    max_rows=1
)

feature_names = data.dtype.names[1:]  # Exclude the first column (ID)

print("Number of features:", len(feature_names))
print("Feature names:", feature_names)

In [ ]:
print("X_train:")
x_train[:1]


print(y_train[:10])
print(np.where(y_train == 1)[0].shape)
print(np.where(y_train == 0)[0].shape)
print(np.where(y_train == -1)[0].shape)
print(y_train.shape)

In [ ]:
data = np.genfromtxt(
    "./dataset/x_train.csv",
    delimiter=",",
    names=True,
    max_rows=1
)

feature_names = data.dtype.names[1:]  # Exclude the first column (ID)

print("Number of features:", len(feature_names))
print("Feature names:", feature_names)

In [ ]:
print("Number of columns:", x_train.shape[1])
print("Rows:", x_train.shape[0])
print("Columns:", x_train.shape[1])
print("Target distribution:")
print(np.unique(y_train, return_counts=True))

In [ ]:
nan_mask = np.isnan(x_train)

nan_count = nan_mask.sum()
total_values = x_train.size
nan_percentage = 100 * nan_count / total_values

print("Number of NaN values:", nan_count)
print("Percentage of NaN values:", nan_percentage, "%")

print("Number of Inf values:", np.isinf(x_train).sum())


nan_per_column = np.isnan(x_train).sum(axis=0)
percentage_per_column = 100 * nan_per_column / x_train.shape[0]

print("Columns with >50% NaN:", np.sum(percentage_per_column > 50))
print("Columns with >90% NaN:", np.sum(percentage_per_column > 90))
print("Maximum NaN percentage:", np.max(percentage_per_column))
print("Minimum NaN percentage:", np.min(percentage_per_column))
#print(percentage_per_column)


### Looking at percentage of Nan in dataset

In [ ]:
for column, percentage in enumerate(percentage_per_column):
    if percentage > 0:
        print(f"Column {column}: {percentage:.2f}% NaN")

for column, percentage in enumerate(percentage_per_column):
    if percentage > 90:
        print(f"Column {column}: {percentage:.2f}% NaN")

columns_over_80 = np.sum(percentage_per_column > 80)
print("Number of columns with more than 80% NaN:", columns_over_80)

columns_over_90 = np.sum(percentage_per_column > 90)
print("Number of columns with more than 90% NaN:", columns_over_90)

columns_0 = np.sum(percentage_per_column == 0)
print("Number of columns with more than 0% NaN:", columns_0)

columns_over_95 = np.sum(percentage_per_column > 95)
print("Number of columns with more than 95% NaN:", columns_over_95)

for i in range(len(feature_names)):
    if percentage_per_column[i] > 90:
        print(i, feature_names[i], percentage_per_column[i])

### Filter the training set so we only keep columns that we have more than 90% 

In [ ]:
# Keep columns with less than 90% NaN values
columns_to_keep = percentage_per_column <= 90

x_train_filtered = x_train[:, columns_to_keep]
x_test_filtered = x_test[:, columns_to_keep]

print("Original shape:", x_train.shape)
print("Filtered shape:", x_train_filtered.shape)
print("Test shape:", x_test_filtered.shape)

nan_filtered = np.isnan(x_train_filtered).sum()
total_filtered = x_train_filtered.size

print("Remaining NaN:", nan_filtered)
print("Remaining NaN %:", 100 * nan_filtered / total_filtered)

In [ ]:
selected_columns = np.where(percentage_per_column <= 90)[0]
print(selected_columns)

In [ ]:
column_names = np.genfromtxt(
    "./dataset/x_train.csv",
    delimiter=",",
    names=True,
    max_rows=1
).dtype.names

selected_columns = np.where(percentage_per_column <= 90)[0]

for index in selected_columns:
    print(index, column_names[index])

data processing, selecting the features

In [1]:
import numpy as np
from helpers import *
from implementations import *
DATA_PATH = "./dataset/"

In [2]:
#get data
print("Loading data...")
x_train, x_test, y_train, train_ids, test_ids = load_csv_data(DATA_PATH)
print("Data loaded")

print("Negative class rate:", np.mean(y_train == -1))
print("Positive class rate:", np.mean(y_train == 1))

Loading data...
Data loaded
Negative class rate: 0.911697929205967
Positive class rate: 0.08830207079403295


In [3]:
X_full, X_test_proc =  preprocess_data(
    x_train, x_test, nan_threshold=0.90
)

--- Starting Preprocessing ---
Number of removed columns with >90% NaNs: 99
Remaining features: 222
Missing values replaced using medians from the training set.
Features standardized (zero mean, unit variance).
Final processed training shape: (328135, 223)
Final processed testing shape: (109379, 223)
--- Preprocessing Complete ---



In [5]:
# Use training data only
correlation = np.corrcoef(X_full, rowvar=False) # Compute correlation matrix

threshold = 0.95 
to_remove = set() # Indices of features to remove

n_features = correlation.shape[0]

for i in range(n_features):
    for j in range(i + 1, n_features):
        if abs(correlation[i, j]) > threshold: # If correlation is above threshold
            to_remove.add(j) # Add feature to remove

selected_indices = [
    i for i in range(n_features)
    if i not in to_remove
]

x_train_selected = X_full[:, selected_indices]
x_test_selected = X_test_proc[:, selected_indices]

print("Original features:", X_full.shape[1])
print("Selected features:", x_train_selected.shape[1])

/opt/anaconda3/envs/project1-grading/lib/python3.9/site-packages/numpy/lib/function_base.py:2853: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/opt/anaconda3/envs/project1-grading/lib/python3.9/site-packages/numpy/lib/function_base.py:2854: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Original features: 223
Selected features: 198
